In [ ]:
#------------------------------------------------ Begin_Librairie ----------------------------------------
import pandas as pd
from bs4 import BeautifulSoup
import re

import requests


import datetime
from selenium import webdriver
from time import sleep
import os


import urllib3

# Disable SSL warnings and skip certificate verification because the server certificate
# cannot be verified in this environment.
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)



In [2]:
#------------------------------------------------ Begin_ fileName ----------------------------------------
regulatorName = 'SY CBSYR' ## change to current controller name

print(f"Running {regulatorName} Web Scraping Tool v.1.1")
now=datetime.datetime.now()
filename = '{} SQL Ready {}.xlsx'.format(regulatorName, str(now).replace(":",".")[:-7])

scriptfolder = f"C:\\Users\\wuj1\\OneDrive - Moody's\\Desktop\\Regulator\\{regulatorName}"

#scriptfolder=os.path.dirname(os.path.abspath(__file__)) ## to decomment for the production environment
os.chdir(scriptfolder)
tempfolder=os.path.join(scriptfolder, 'tempfolder') #if files are downloaded during the process

if os.path.exists(tempfolder):
    for rem in os.listdir(tempfolder):
        os.remove(os.path.join(tempfolder, rem))
else:
    os.mkdir(tempfolder)


Running SY CBSYR Web Scraping Tool v.1.1


In [3]:
#------------------------------------------------ Begin_chromedriver ----------------------------------------
#Starting Chrome driver, set to download files in tempfolder
chromeOptions = webdriver.ChromeOptions()
prefs = {"plugins.always_open_pdf_externally": True,
		 "download.prompt_for_download": False,
		 "download.default_directory" : tempfolder,
         'profile.default_content_setting_values.automatic_downloads': 1 # Desable a Multiplefile download alert
         }
chromeOptions.add_experimental_option("prefs",prefs)
driver = webdriver.Chrome(options=chromeOptions)
driver.maximize_window()


In [4]:
#------------------------------------------------ Begin_Variable ----------------------------------------


sqldict={'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 
          'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 
          'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 
          'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],
          'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 
          'Phone - Mother company': [], 'Check': []}

processdate = now.strftime('%Y-%m-%d')

# %%

#------------------------------------------------ Begin_Variable ----------------------------------------

regdict={

        regulatorName + ' 1': 'https://cb.gov.sy/index.php?page=show&ex=2&dir=items&lang=2&ser=1&cat_id=1372&act=1372&',
        regulatorName + ' 2': 'https://cb.gov.sy/index.php?page=show&ex=2&dir=items&lang=2&ser=1&cat_id=1374&act=1374&',

        }



Typology={

       regulatorName + ' 1': 'Banks',
       regulatorName + ' 2': 'Microfinance Banks',


        }

In [5]:
#------------------------------------------------ Begin_Fouction ----------------------------------------
def bourange_same_length_array(sqldict) :
    maxlen = len(sqldict['ListProcessDate'])
    for key, val in sqldict.items():
        if len(sqldict[key]) != maxlen:
            empty = []
            total_empty = maxlen - len(sqldict[key])
            for i in range(total_empty):
                empty.append('')
            sqldict[key]=sqldict[key]+empty
    return sqldict

def clean_candidate_name(text):
    text = re.sub(r'\\s+', ' ', text).strip(' -:|\\t\\r\\n')
    if not text:
        return ''

    lowered = text.lower()
    blacklist = {
        'home', 'more', 'details', 'read more', 'back', 'next', 'previous',
        'banks', 'microfinance banks'
    }
    if lowered in blacklist:
        return ''

    if len(text) < 3 or len(text) > 200:
        return ''

    return text

def extract_bank_names_from_listing(soup):
    names = []
    seen = set()

    def add_name(raw_text):
        name = clean_candidate_name(raw_text)
        if not name:
            return
        normalized = name.casefold()
        if normalized in seen:
            return
        seen.add(normalized)
        names.append(name)

    for spage in soup.select('div.spage'):
        for node in spage.select('div.spDesc a.SubDoc'):
            add_name(node.get_text(' ', strip=True).replace('•', ' '))

    if names:
        return names

    for node in soup.select('a.SubDoc'):
        add_name(node.get_text(' ', strip=True).replace('•', ' '))

    return names


In [6]:
#------------------------------------------------ Begin_Main ----------------------------------------

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36'
}

for k, reg in enumerate(regdict):
    print(f"[INFO] : Working {k+1}/{len(regdict)} _({reg})_ ")
    page_html = ''
    try:
        response = requests.get(regdict[reg], headers=headers, timeout=30, verify=False)
        response.raise_for_status()
        if '\\ufffd' in response.text and response.apparent_encoding:
            response.encoding = response.apparent_encoding
        page_html = response.text
    except requests.RequestException as exc:
        print(f"[WARN] : Request failed for {reg}: {exc}")
        print(f"[INFO] : Falling back to Selenium for {reg}")
        try:
            driver.get(regdict[reg])
            sleep(3)
            page_html = driver.page_source
        except Exception as selenium_exc:
            print(f"[WARN] : Selenium fallback failed for {reg}: {selenium_exc}")
            continue

    soup = BeautifulSoup(page_html, 'html.parser')
    bank_names = extract_bank_names_from_listing(soup)

    if not bank_names:
        print(f"[WARN] : No bank names found for {reg}")
        continue

    for bank_name in bank_names:
        sqldict['ListProcessDate'].append(processdate)
        sqldict['RegCtry'].append(reg.split()[0])
        sqldict['RegCode'].append(reg.split()[1])
        sqldict['ListCode'].append(reg.split()[2])
        sqldict['RegulationType'].append('Regulated')
        sqldict['ListName'].append(Typology[reg])
        sqldict['Name'].append(bank_name)
        sqldict = bourange_same_length_array(sqldict)

    print(f"[INFO] : Extracted {len(bank_names)} bank names from {reg}")


[INFO] : Working 1/2 _(SY CBSYR 1)_ 
[WARN] : Request failed for SY CBSYR 1: HTTPSConnectionPool(host='cb.gov.sy', port=443): Max retries exceeded with url: /index.php?page=show&ex=2&dir=items&lang=2&ser=1&cat_id=1372&act=1372& (Caused by SSLError(SSLError(1, '[SSL: SSLV3_ALERT_HANDSHAKE_FAILURE] sslv3 alert handshake failure (_ssl.c:1000)')))
[INFO] : Falling back to Selenium for SY CBSYR 1
[INFO] : Extracted 20 bank names from SY CBSYR 1
[INFO] : Working 2/2 _(SY CBSYR 2)_ 
[WARN] : Request failed for SY CBSYR 2: HTTPSConnectionPool(host='cb.gov.sy', port=443): Max retries exceeded with url: /index.php?page=show&ex=2&dir=items&lang=2&ser=1&cat_id=1374&act=1374& (Caused by SSLError(SSLError(1, '[SSL: SSLV3_ALERT_HANDSHAKE_FAILURE] sslv3 alert handshake failure (_ssl.c:1000)')))
[INFO] : Falling back to Selenium for SY CBSYR 2
[INFO] : Extracted 4 bank names from SY CBSYR 2


In [7]:
#------------------------------------------------ Begin_writer and save df to excel  ----------------------------------------
os.chdir(scriptfolder)
df=pd.DataFrame(sqldict)
df.to_excel(filename, index=False)

driver.quit()
sleep(3)

In [8]:
df

,bvdid,priority,ListLabel,Typology,EntryType,Name,InternalID_1,InternalID_1_type,InternalID_2,InternalID_2_type,...,LEI Code,BIC SWIFT Code,Name - Mother Company,Address_1 - Mother company,Address_2 - Mother company,City - Mother company,Zip - Mother company,Cntry - Mother company,Phone - Mother company,Check
0,,,,,,Commercial Bank of Syria,,,,,...,,,,,,,,,,
1,,,,,,Popular Credit Bank,,,,,...,,,,,,,,,,
2,,,,,,The Saving Bank,,,,,...,,,,,,,,,,
3,,,,,,Agricultural Cooperative Bank,,,,,...,,,,,,,,,,
4,,,,,,Industrial Bank,,,,,...,,,,,,,,,,
5,,,,,,Real Estate Bank,,,,,...,,,,,,,,,,
6,,,,,,Banque Bemo Saudi Fransi,,,,,...,,,,,,,,,,
7,,,,,,Syria Gulf Bank,,,,,...,,,,,,,,,,
8,,,,,,The International Bank for Trade and Finance,,,,,...,,,,,,,,,,
9,,,,,,Bank of Syria and Overseas,,,,,...,,,,,,,,,,
